In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Complete DistilBERT Fine‑tuning for LIAR

In [ ]:
!pip install transformers datasets scikit-learn -q

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # Import AdamW from torch.optim, not transformers
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import requests
import zipfile
import io
import re

In [ ]:
print("Downloading LIAR dataset...")
url = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"

response = requests.get(url, timeout=60)
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    train_df = pd.read_csv(z.open('train.tsv'), sep='\t', header=None)
    test_df = pd.read_csv(z.open('test.tsv'), sep='\t', header=None)
    valid_df = pd.read_csv(z.open('valid.tsv'), sep='\t', header=None)

column_names = ['id', 'label', 'statement', 'subject', 'speaker', 'job', 'state', 'party',
                'barely_true_counts', 'false_counts', 'half_true_counts',
                'mostly_true_counts', 'pants_on_fire_counts', 'context']

for df in [train_df, test_df, valid_df]:
    df.columns = column_names[:len(df.columns)]

# Map string labels to binary (0=Fake, 1=True)
def to_binary(label):
    if label in ['pants-fire', 'false', 'barely-true']:
        return 0  # Fake
    else:
        return 1  # True

train_df['label_binary'] = train_df['label'].apply(to_binary)
test_df['label_binary'] = test_df['label'].apply(to_binary)
valid_df['label_binary'] = valid_df['label'].apply(to_binary)

print(f"Train: {len(train_df)}, Test: {len(test_df)}, Valid: {len(valid_df)}")
print(f"Train label distribution: Fake={sum(train_df['label_binary']==0)}, True={sum(train_df['label_binary']==1)}")

In [ ]:
# Combine train + validation for training
train_texts = pd.concat([train_df['statement'], valid_df['statement']]).values
train_labels = pd.concat([train_df['label_binary'], valid_df['label_binary']]).values

test_texts = test_df['statement'].values
test_labels = test_df['label_binary'].values

print(f"Combined train size: {len(train_texts)}")
print(f"Test size: {len(test_texts)}")

In [ ]:
#Clean text function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_texts = [clean_text(t) for t in train_texts]
test_texts = [clean_text(t) for t in test_texts]
print("Text cleaning complete.")

In [ ]:
#Create PyTorch Dataset
class LiarDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding='max_length', 
                                   max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Create datasets
train_dataset = LiarDataset(train_texts, train_labels, tokenizer)
test_dataset = LiarDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

In [ ]:
#Load DistilBERT model and train
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

# Training loop
epochs = 4
for epoch in range(epochs):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        loop.set_postfix(loss=loss.item())
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Evaluating'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"\n{'='*50}")
print(f"DISTILBERT ON LIAR TEST SET")
print(f"{'='*50}")
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Fake', 'True']))

# testing on kaggle set

In [ ]:
#DistilBERT on fake news dataset
# Test DistilBERT on your original Kaggle test set
import joblib

# Load Kaggle test data
_, X_test_kaggle, _, y_test_kaggle = joblib.load('/kaggle/working/data_splits.pkl')

# Clean Kaggle texts
X_test_kaggle_clean = [clean_text(t) for t in X_test_kaggle]

# Create dataset and evaluate
kaggle_dataset = LiarDataset(X_test_kaggle_clean, y_test_kaggle.values, tokenizer)
kaggle_loader = DataLoader(kaggle_dataset, batch_size=32, shuffle=False)

model.eval()
all_preds = []
with torch.no_grad():
    for batch in tqdm(kaggle_loader, desc='Kaggle Test'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())

acc_kaggle = accuracy_score(y_test_kaggle.values, all_preds)
print(f"\n{'='*50}")
print(f"DISTILBERT ON KAGGLE TEST SET")
print(f"{'='*50}")
print(f"Accuracy: {acc_kaggle:.4f} ({acc_kaggle*100:.2f}%)")

# Train DistilBERT on Combined Kaggle + LIAR Data

In [ ]:
import joblib

# Load Kaggle training data (clean text and labels)
X_train_kaggle, X_test_kaggle, y_train_kaggle, y_test_kaggle = joblib.load('/kaggle/working/data_splits.pkl')

# Convert to lists for combining
kaggle_texts = X_train_kaggle.tolist()[:8000]  # Use 8000 samples (balance with LIAR)
kaggle_labels = y_train_kaggle.tolist()[:8000]

print(f"Kaggle samples: {len(kaggle_texts)}")
print(f"Kaggle label distribution: Fake={kaggle_labels.count(0)}, True={kaggle_labels.count(1)}")

In [ ]:
# Use LIAR train texts and labels from earlier (already in memory)
liar_texts = train_texts[:4000]  # list, so no .tolist()
liar_labels = train_labels[:4000]  # numpy array

print(f"LIAR samples: {len(liar_texts)}")
print(f"LIAR label distribution: Fake={np.sum(liar_labels == 0)}, True={np.sum(liar_labels == 1)}")

In [ ]:
# Convert numpy arrays to lists before combining
kaggle_labels_list = kaggle_labels.tolist() if hasattr(kaggle_labels, 'tolist') else kaggle_labels
liar_labels_list = liar_labels.tolist() if hasattr(liar_labels, 'tolist') else liar_labels

# Combine Kaggle + LIAR
combined_texts = kaggle_texts + liar_texts
combined_labels = kaggle_labels_list + liar_labels_list

print(f"Combined training size: {len(combined_texts)}")
print(f"Combined label distribution: Fake={combined_labels.count(0)}, True={combined_labels.count(1)}")

In [ ]:
# Recreate dataset and dataloader with combined data
class CombinedDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding='max_length',
                                   max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create combined dataset
combined_dataset = CombinedDataset(combined_texts, combined_labels, tokenizer)
combined_loader = DataLoader(combined_dataset, batch_size=16, shuffle=True)

print(f"Combined loader: {len(combined_loader)} batches")

In [ ]:
# Load fresh model (not the one that forgot Kaggle)
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

# Train for 3-4 epochs on combined data
epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    loop = tqdm(combined_loader, desc=f'Epoch {epoch+1}/{epochs}')
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        loop.set_postfix(loss=loss.item())
    
    avg_loss = total_loss / len(combined_loader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

In [ ]:
# Test on Kaggle test set
def evaluate_on_kaggle():
    kaggle_test_clean = [clean_text(t) for t in X_test_kaggle]
    kaggle_dataset = CombinedDataset(kaggle_test_clean, y_test_kaggle.values, tokenizer)
    kaggle_loader = DataLoader(kaggle_dataset, batch_size=32, shuffle=False)
    
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(kaggle_loader, desc='Kaggle Test'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
    
    acc = accuracy_score(y_test_kaggle.values, all_preds)
    return acc

# Test on LIAR test set
def evaluate_on_liar():
    liar_test_clean = [clean_text(t) for t in test_texts]
    liar_dataset = CombinedDataset(liar_test_clean, test_labels, tokenizer)
    liar_loader = DataLoader(liar_dataset, batch_size=32, shuffle=False)
    
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(liar_loader, desc='LIAR Test'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    return acc

kaggle_acc = evaluate_on_kaggle()
liar_acc = evaluate_on_liar()

print(f"\n{'='*50}")
print(f"COMBINED TRAINING RESULTS")
print(f"{'='*50}")
print(f"Kaggle Test Accuracy: {kaggle_acc:.4f} ({kaggle_acc*100:.2f}%)")
print(f"LIAR Test Accuracy: {liar_acc:.4f} ({liar_acc*100:.2f}%)")